In [3]:
# -------------------------------------------------
# 0️⃣  Core libraries
# -------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Use CPU (change to "cuda" if you have a GPU)
device = torch.device("cpu")

In [4]:
# -------------------------------------------------
# 1️⃣  A toy cooperative defense environment
#     - Each agent sees only its own 1‑D position
#     - Shared reward = - (sum of absolute positions)
# -------------------------------------------------
class SimpleDefenseEnv:
    def __init__(self, n_agents: int = 2):
        self.n_agents = n_agents
        self.obs_dim = 1               # each agent observes its own position
        self.action_dim = 1            # continuous action in [-1, 1] (after tanh)

    def reset(self):
        """Randomly initialise agents in [-5, 5]"""
        self.positions = np.random.uniform(-5, 5, size=self.n_agents)
        # Return a list of observations, one per agent
        return [np.array([p]) for p in self.positions]

    def step(self, actions):
        """
        actions: list/array of shape (n_agents, 1) – raw actor output (already bounded)
        Returns: next_obs, rewards, done
        """
        # Apply actions
        for i in range(self.n_agents):
            self.positions[i] += float(actions[i][0])

        # Shared penalty: distance from origin
        dist_penalty = np.sum(np.abs(self.positions))
        reward = -dist_penalty                     # the more spread out, the worse
        rewards = [reward for _ in range(self.n_agents)]

        # Next observations
        next_obs = [np.array([p]) for p in self.positions]
        done = False                               # episodic task not used here
        return next_obs, rewards, done

In [5]:
# -------------------------------------------------
# 2️⃣  Actor: maps local observation → action
#     Uses Tanh to keep actions in [-1, 1]
# -------------------------------------------------
class Actor(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()                     # bounds the output
        )

    def forward(self, obs):
        return self.net(obs)

In [6]:
# -------------------------------------------------
# 3️⃣  Centralised Critic: sees the *joint* observation
#     and *joint* action of all agents → Q‑value
# -------------------------------------------------
class CentralizedCritic(nn.Module):
    def __init__(self, total_obs_dim: int, total_action_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(total_obs_dim + total_action_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)               # scalar Q-value
        )

    def forward(self, joint_obs, joint_actions):
        """
        joint_obs:   (batch, total_obs_dim)
        joint_actions: (batch, total_action_dim)
        """
        x = torch.cat([joint_obs, joint_actions], dim=-1)
        return self.net(x)

In [7]:
# -------------------------------------------------
# 4️⃣  Agent holds an actor + a centralised critic
#     plus their respective optimizers
# -------------------------------------------------
class MADDPGAgent:
    def __init__(self,
                 obs_dim: int,
                 action_dim: int,
                 total_obs_dim: int,
                 total_action_dim: int,
                 lr: float = 1e-3):
        # Actor (decentralised)
        self.actor = Actor(obs_dim, action_dim).to(device)
        # Critic (centralised)
        self.critic = CentralizedCritic(total_obs_dim, total_action_dim).to(device)

        # Optimizers
        self.actor_optim  = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=lr)

    def select_action(self, obs, noise_scale: float = 0.1):
        """
        Returns a noisy action for exploration.
        obs: numpy array of shape (obs_dim,)
        """
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)  # (1, obs_dim)
        action = self.actor(obs_t).detach().numpy()[0]              # (action_dim,)
        # Add Gaussian exploration noise
        action += noise_scale * np.random.randn(*action.shape)
        # Clip to valid range [-1, 1]
        return np.clip(action, -1, 1)

In [8]:
# -------------------------------------------------
# 5️⃣  MADDPG training routine
# -------------------------------------------------
def train_maddpg(num_episodes: int = 200,
                 gamma: float = 0.95):
    """
    Returns a list of trained agents (one per environment agent).
    """
    env = SimpleDefenseEnv(n_agents=2)

    n_agents   = env.n_agents
    obs_dim    = env.obs_dim
    action_dim = env.action_dim
    total_obs_dim    = obs_dim * n_agents
    total_action_dim = action_dim * n_agents

    # One agent (actor+critic) per environment agent
    agents = [
        MADDPGAgent(obs_dim, action_dim, total_obs_dim, total_action_dim)
        for _ in range(n_agents)
    ]

    mse_loss = nn.MSELoss()

    for episode in range(num_episodes):
        # ---- 1️⃣  Reset environment & get local observations ----
        obs = env.reset()                     # list length n_agents, each np.array([pos])

        # ---- 2️⃣  Each agent selects an action using ONLY its obs ----
        actions = [agents[i].select_action(obs[i]) for i in range(n_agents)]

        # ---- 3️⃣  Step the environment with the joint action ----
        next_obs, rewards, done = env.step(actions)

        # ---- 4️⃣  Build joint (global) tensors for the critic ----
        joint_obs     = torch.tensor(np.concatenate(obs),     dtype=torch.float32).unsqueeze(0)   # (1, total_obs_dim)
        joint_next_obs= torch.tensor(np.concatenate(next_obs),dtype=torch.float32).unsqueeze(0)
        joint_actions = torch.tensor(np.concatenate(actions), dtype=torch.float32).unsqueeze(0)   # (1, total_action_dim)

        # ---- 5️⃣  Compute target Q‑value using next‑state actors ----
        next_actions = []
        for i in range(n_agents):
            next_obs_i = torch.tensor(next_obs[i], dtype=torch.float32).unsqueeze(0)
            next_actions.append(agents[i].actor(next_obs_i))          # each (1, action_dim)
        joint_next_actions = torch.cat(next_actions, dim=-1)          # (1, total_action_dim)

        # ---- 6️⃣  Update each agent's centralized critic ----
        for i in range(n_agents):
            with torch.no_grad():
                target_q = agents[i].critic(joint_next_obs, joint_next_actions)   # (1,1)
                y = rewards[i] + gamma * target_q                                   # TD target

            current_q = agents[i].critic(joint_obs, joint_actions)                # (1,1)
            critic_loss = mse_loss(current_q, y)

            agents[i].critic_optim.zero_grad()
            critic_loss.backward()
            agents[i].critic_optim.step()

        # ---- 7️⃣  Update each agent's actor (policy gradient) ----
        for i in range(n_agents):
            obs_i = torch.tensor(obs[i], dtype=torch.float32).unsqueeze(0)       # (1, obs_dim)
            predicted_action_i = agents[i].actor(obs_i)                         # (1, action_dim)

            # Build joint action vector where agent i uses its *predicted* action,
            # others use the actions actually taken in the environment.
            actions_for_grad = []
            for j in range(n_agents):
                if j == i:
                    actions_for_grad.append(predicted_action_i)
                else:
                    a_j = torch.tensor(actions[j], dtype=torch.float32).unsqueeze(0)
                    actions_for_grad.append(a_j)
            joint_actions_grad = torch.cat(actions_for_grad, dim=-1)   # (1, total_action_dim)

            # Actor loss = -Q (we want to maximise Q)
            actor_loss = -agents[i].critic(joint_obs, joint_actions_grad).mean()

            agents[i].actor_optim.zero_grad()
            actor_loss.backward()
            agents[i].actor_optim.step()

        # ---- 8️⃣  Logging ----
        if (episode + 1) % 20 == 0:
            avg_reward = np.mean(rewards)
            print(f"Episode {episode+1}/{num_episodes} | Avg Reward: {avg_reward:.3f}")

    return agents

In [9]:
# -------------------------------------------------
# 6️⃣  Execute training and print a short recap
# -------------------------------------------------
if __name__ == "__main__":
    trained_agents = train_maddpg(num_episodes=200)

    print("\nTraining complete. Each agent now has:")
    print(" - A decentralized Actor (local obs → action)")
    print(" - A centralized Critic (global obs+actions → Q‑value)")

Episode 20/200 | Avg Reward: -5.368
Episode 40/200 | Avg Reward: -1.701
Episode 60/200 | Avg Reward: -3.727
Episode 80/200 | Avg Reward: -3.840
Episode 100/200 | Avg Reward: -0.684
Episode 120/200 | Avg Reward: -6.241
Episode 140/200 | Avg Reward: -4.886
Episode 160/200 | Avg Reward: -0.455
Episode 180/200 | Avg Reward: -5.764
Episode 200/200 | Avg Reward: -5.071

Training complete. Each agent now has:
 - A decentralized Actor (local obs → action)
 - A centralized Critic (global obs+actions → Q‑value)
